# RLVR with a Custom Reward Function

## Lab 4 - Evaluate the customized model

In this lab, you will evaluate the RLVR fine-tuned model against the **MATH benchmark** and compare it to the base model to measure the improvement in mathematical reasoning.

### Why benchmark evaluation?

RLVR trains models on tasks with objectively verifiable answers, so it makes sense to evaluate them the same way — using a standardized benchmark with known correct answers rather than subjective LLM-as-a-Judge scoring.

### What you'll do in this notebook

1. Retrieve the fine-tuned model from the Model Registry
2. Explore available benchmarks and create a `BenchMarkEvaluator`
3. Run evaluation on both the fine-tuned and base model
4. Compare the results

---

## 1. Prerequisites and SageMaker setup

In [ ]:
%pip install -r requirements.txt

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

base_model_id = "huggingface-reasoning-qwen3-06b"
project_prefix = "gsm8k-custom-reward-rlvr"

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sess.boto_region_name}")

---

## 2. Retrieve the custom-reward fine-tuned model

This mirrors Lab 3: we locate the latest model package in the model package group created by the training notebook. If you want to evaluate a specific model package version, set `fine_tuned_model_package_arn` manually before running the lookup cell.

In [ ]:
model_package_group_name = f"{base_model_id}-custom-reward-rlvr"
fine_tuned_model_package_arn = None

if fine_tuned_model_package_arn is None:
    response = sm_client.list_model_packages(
        ModelPackageGroupName=model_package_group_name,
        SortBy="CreationTime",
        SortOrder="Descending",
        MaxResults=1,
    )
    fine_tuned_model_package_arn = response["ModelPackageSummaryList"][0]["ModelPackageArn"]

if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{project_prefix}/evaluation"
else:
    output_path = f"s3://{bucket_name}/{project_prefix}/evaluation"

print(f"Model Package Group: {model_package_group_name}")
print(f"Fine-tuned Model Package ARN: {fine_tuned_model_package_arn}")
print(f"Evaluation output path: {output_path}")

---

## 3. Explore available benchmarks

SageMaker provides several built-in benchmarks. As in Lab 3, this lab uses the **MATH** benchmark for mathematical reasoning evaluation.

In [ ]:
from sagemaker.train.evaluate import BenchMarkEvaluator, get_benchmarks, get_benchmark_properties
from rich.pretty import pprint

Benchmark = get_benchmarks()
pprint(list(Benchmark))

In [ ]:
pprint(get_benchmark_properties(benchmark=Benchmark.MATH))

---

## 4. Create and run the benchmark evaluator

This cell intentionally follows the Lab 3 `BenchMarkEvaluator` pattern: pass the benchmark enum (`Benchmark.MATH`) and use `s3_output_path` for results.

> **Expected duration:** benchmark evaluation can take 15-30 minutes depending on service capacity.

In [ ]:
evaluator = BenchMarkEvaluator(
    benchmark=Benchmark.MATH,
    model=fine_tuned_model_package_arn,
    model_package_group=model_package_group_name,
    base_eval_name="custom-reward-rlvr",
    s3_output_path=output_path,
    evaluate_base_model=False,
    role=role,
    sagemaker_session=sess,
)

execution = evaluator.evaluate()
execution.wait()
print(execution)

---

## 5. View evaluation results

We retrieve the latest succeeded benchmark evaluation and display the results. The output shows the MATH benchmark scores for both the base model and the RLVR fine-tuned model side by side.

In [ ]:
from sagemaker.train.evaluate import EvaluationPipelineExecution
from sagemaker.train.evaluate.constants import EvalType
from rich.pretty import pprint


# Get all succeeded evaluations and take the first 2
all_succeeded = [
    e for e in EvaluationPipelineExecution.get_all(eval_type=EvalType.BENCHMARK)
    if e.status.overall_status == "Succeeded"
]

last_succeeded = all_succeeded[:1]

# Print both
for i, execution in enumerate(last_two_succeeded, 1):
    print(f"=== Succeeded Evaluation #{i} ===")
    pprint(execution)
    pprint(execution.show_results())